# Small language models for translation
## A tiny Transformer from scratch and chrF minimum Bayes risk decoding

**EN.737 · Semester project · M2 and Q3(a)**

You will train a decoder-only model to finish a bilingual prompt, then hold its weights fixed and change how it chooses a translation. The final table and plot answer *how much translation quality changes for each extra millisecond spent decoding*. All numerical results are calculated by the cells.

**First run in Colab.** Choose **Runtime → Change runtime type → GPU** and run the notebook from top to bottom. The default project run downloads German–English IWSLT 2017 data. To check the wiring quickly, set `PROFILE = "smoke"` in the settings cell. For your team's experiment, choose the agreed pair, prompt, and splits **before running the project profile**. The data section explains the three-file TSV format.

| Stage | What to look for |
|---|---|
| Prepare parallel text | Fixed split IDs, example prompt and answer, train-only tokenizer, and length coverage |
| Train M2 | Exact parameter count, tokens processed, supervised target tokens, and checkpoint selected by validation loss |
| Compare decoders | Greedy, beam, and sample-and-rerank on the same frozen checkpoint and sources, with repeated measurements |
| Interpret the result | Quality–latency plot, paired comparisons, translations to inspect, and a live M2 demo |

The project brief leaves the language pair open. German → English is one of its suggested choices. If teammates use M0 or M1, give them the same evaluation IDs, raw references, and prompt format. M2 learns its own tokenizer and all model weights from the parallel training data. Across different tokenizers, corpus BLEU and chrF provide a common unit for comparison. A unit of token cross-entropy depends on the tokenizer.

### Prepare the Colab runtime

PyTorch supplies tensors, gradients, and GPU kernels. The notebook defines the Transformer itself. `tokenizers` learns the byte-pair vocabulary, `pyarrow` reads the official data files, and `sacrebleu` supplies both corpus metrics and the sentence-level reranking utility. This cell installs missing packages. The settings cell records the versions actually used.

In [ ]:
from importlib import metadata, util
import subprocess
import sys

fallback_versions = {"tokenizers": "0.21.4", "sacrebleu": "2.5.1", "pyarrow": "20.0.0"}
missing = [f"{name}=={version}" for name, version in fallback_versions.items()
           if util.find_spec(name) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])

import hashlib
import json
import math
import os
import platform
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sacrebleu.metrics import BLEU, CHRF

### Choose the dataset, training budget, and decoding experiment

The **project** profile starts with up to 20,000 training pairs and six passes through the retained set. The data cells report how many pairs actually survive filtering. The **smoke** profile learns a small vocabulary of names, verbs and foods so you can inspect actual translations on CPU. It takes 32 short passes over 72 synthetic training pairs. Its scores have no scientific interpretation.

For your team's data, set `DATA_MODE = "tsv"`, set the language names and `DATA_DIR`, and place `train.tsv`, `validation.tsv`, and `test.tsv` there. In Colab you can create the folder and upload the files from the **Files** sidebar, or use a mounted Google Drive folder. Edit `prompt_template` if your team agrees on a different prompt. `context_len` limits token positions in the model. `max_new_tokens` is the shared stopping limit for all decoders. The vocabulary size is an upper target for the BPE fit.

The project run uses **one training seed and three decoding seeds**. You can add training seeds to study M2 training variability. Q3 always decodes the checkpoint for the *first listed training seed* so that a change in translation can be attributed to the decoder. The model section reports token budgets separately for each seed.

Colab's default output directory is local to its runtime. To keep checkpoints through a disconnect, set `USE_DRIVE = True` before starting. All outputs are placed beneath the run folder printed below.

In [ ]:
PROFILE = "project"                  # "project" or "smoke"
DATA_MODE = "iwslt"                  # project mode: "iwslt" or "tsv"
SOURCE_LANGUAGE = "German"
TARGET_LANGUAGE = "English"
DATA_DIR = "/content/team_data" if Path("/content").is_dir() else "./team_data"
USE_DRIVE = False

if PROFILE not in {"project", "smoke"} or DATA_MODE not in {"iwslt", "tsv"}:
    raise ValueError("Choose one of the profile and data mode options listed above.")
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RUNS = Path("/content/drive/MyDrive/en737_m2")
else:
    home_for_runs = Path("/content") if Path("/content").is_dir() else Path.cwd()
    RUNS = home_for_runs / "en737_m2_runs"

CFG = {
    "data_mode": DATA_MODE, "data_dir": DATA_DIR,
    "source_lang": SOURCE_LANGUAGE, "target_lang": TARGET_LANGUAGE,
    "prompt_template": "{source_lang}: {source}\n{target_lang}:",
    "split_seed": 737, "max_train": 20_000, "max_validation": 512, "max_test": 512,
    "vocab_size": 4096, "context_len": 192, "max_new_tokens": 64,
    "d_model": 192, "n_heads": 4, "n_layers": 4, "d_ff": 768, "dropout": 0.1,
    "batch_size": 32, "epochs": 6, "lr": 0.0005, "weight_decay": 0.01,
    "grad_clip": 1.0, "train_seeds": [17],
    "decode_seeds": [101, 202, 303], "eval_sources": 64, "eval_subset_seed": 8021,
    "k_values": [1, 2, 4, 8], "beam_width": 4, "length_penalty": 0.6,
}
if PROFILE == "smoke":
    CFG.update(data_mode="smoke", source_lang="German", target_lang="English",
               max_train=None, max_validation=None, max_test=None, vocab_size=384,
               context_len=80, max_new_tokens=12, d_model=64, n_heads=4, n_layers=2,
               d_ff=128, batch_size=12, epochs=32, lr=0.002, dropout=0.0, eval_sources=4,
               k_values=[1, 2, 4], beam_width=3, decode_seeds=[101, 202])

if not CFG["train_seeds"] or not CFG["decode_seeds"]:
    raise ValueError("Supply at least one training seed and one decoding seed.")
for key in ("train_seeds", "decode_seeds", "k_values"):
    if len(set(CFG[key])) != len(CFG[key]):
        raise ValueError(f"{key} must contain distinct entries.")
if 1 not in CFG["k_values"] or len(CFG["k_values"]) < 2 or min(CFG["k_values"]) < 1:
    raise ValueError("Include k=1 and at least one larger sample count.")
if min(CFG["eval_sources"], CFG["beam_width"], CFG["max_new_tokens"]) < 1:
    raise ValueError("Evaluation size, beam width and generation limit must be positive.")
if CFG["max_new_tokens"] >= CFG["context_len"]:
    raise ValueError("Reserve room in the context for the source prompt.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cpu":
    torch.set_num_threads(min(4, torch.get_num_threads()))
versions = {pkg: metadata.version(pkg) for pkg in
            ["torch", "numpy", "pandas", "matplotlib", "pyarrow", "tokenizers", "sacrebleu"]}
hardware = {"device": str(DEVICE), "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None,
            "cuda": torch.version.cuda, "python": platform.python_version(), "platform": platform.platform()}
identity_data = {"config": CFG}
if CFG["data_mode"] == "tsv":
    # Editing a team's split creates a fresh run while keeping earlier results.
    file_hashes = {}
    for split in ("train", "validation", "test"):
        path = Path(DATA_DIR) / f"{split}.tsv"
        if not path.is_file():
            raise FileNotFoundError(f"Place {split}.tsv in {DATA_DIR}, or update DATA_DIR.")
        with path.open("rb") as source_file:
            file_hashes[split] = hashlib.file_digest(source_file, "sha256").hexdigest()
    identity_data["tsv_sha256"] = file_hashes
run_key = hashlib.sha256(json.dumps(identity_data, sort_keys=True).encode()).hexdigest()[:12]
OUT = RUNS / f"{PROFILE}-{run_key}"
OUT.mkdir(parents=True, exist_ok=True)

def write_json(path, obj):
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")

write_json(OUT / "settings.json", {"profile": PROFILE, **identity_data, "versions": versions,
                                  "hardware": hardware})
print(f"Profile: {PROFILE} | Device: {hardware['gpu'] or DEVICE} | Outputs: {OUT}")
if PROFILE == "project" and DEVICE.type == "cpu":
    print("A GPU is recommended for the 20,000-pair run. The smoke profile also works on CPU.")

### Record the prediction before training or scoring

**Q3(a) hypothesis.** Drawing more translations gives chrF reranking a better chance of selecting a candidate that agrees with several plausible translations. **Quantitative prediction:** on the fixed test subset, the largest configured sample pool will improve mean corpus chrF by at least **1.0 point** over a single sample and its mean paired latency ratio (largest pool divided by a single sample) will exceed **1.0**. Greedy and beam search are the deterministic reference methods. MBR can still favor a common, incomplete translation. The prediction can fail.

This cell writes the prediction with the settings, before any translations are scored. The result section computes the observed differences even when they go in the opposite direction. If the team changes the prediction after seeing results, record that change as exploratory.

In [ ]:
plan = {
    "model": "M2, first listed training seed, minimum validation target cross-entropy",
    "contrast": f"chrF-MBR k={max(CFG['k_values'])} versus one ancestral sample",
    "quality_threshold_chrf_points": 1.0, "cost_prediction": "mean paired end-to-end latency ratio > 1.0",
    "controls": ["checkpoint", "source IDs", "prompt", "generation limit", "device",
                 "one source per request", "no key/value cache", "ancestral temperature 1"],
    "repeats": CFG["decode_seeds"], "subset_seed": CFG["eval_subset_seed"],
}
write_json(OUT / "prediction_before_scoring.json", plan)
print(plan["contrast"], "| predicted gain ≥ 1.0 chrF point, with higher cost")

### Load the parallel sentences

The project leaves the language pair open. The `iwslt` setting downloads the **German → English** release, checks the SHA-256 of each file, and keeps the published train, validation, and test boundaries. It does not repartition TED talks. The revision and checksums below fix the exact data version. [IWSLT 2017 files](https://huggingface.co/datasets/IWSLT/iwslt2017/tree/b12a1d9c35c86684cfb15079c9a2fbb467a23aa1/iwslt2017-de-en)

For your team's corpus choose `tsv`, set the two language names and point `data_dir` at a folder with **train.tsv**, **validation.tsv**, and **test.tsv**. Each UTF-8 file needs a header with `source` and `target` separated by a tab. Example first data row: `Guten Morgen!<TAB>Good morning!`, where `<TAB>` represents one tab character. An optional `id` column preserves your team's row IDs.

The `smoke` setting constructs short German–English sentences with fixed splits so you can run the entire workflow before using real data. Scores from this setting only test the code path.

In [ ]:
import csv
import hashlib
import json
import random
import shutil
import tempfile
import unicodedata
import urllib.request
from pathlib import Path

SPLIT_NAMES = ("train", "validation", "test")
IWSLT_REVISION = "b12a1d9c35c86684cfb15079c9a2fbb467a23aa1"
IWSLT_SHA256 = {
    "train": "775169702cc226a66b768ab7f3fb22b744ce5a6e037b677e068a9d51c4af9fce",
    "validation": "6c4385add06f6d19de34978cc481e59a1f1b354ceaa2247101629f15c351b6f3",
    "test": "2ed8e9880200bd7adddd9e56664f8607fb1dbbaff18f00bd811ac8378c4e4b94",
}

def file_sha256(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

def read_iwslt(split):
    import pyarrow.parquet as pq

    folder = Path(CFG["data_dir"]) / "iwslt2017" / IWSLT_REVISION
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / f"iwslt2017-{split}.parquet"
    if not path.exists() or file_sha256(path) != IWSLT_SHA256[split]:
        url = (f"https://huggingface.co/datasets/IWSLT/iwslt2017/resolve/"
               f"{IWSLT_REVISION}/iwslt2017-de-en/{path.name}")
        print(f"Fetching IWSLT {split} ...")
        with tempfile.NamedTemporaryFile(dir=folder, suffix=".part", delete=False) as tmp:
            tmp_path = Path(tmp.name)
        try:
            with urllib.request.urlopen(url, timeout=120) as response, tmp_path.open("wb") as dest:
                shutil.copyfileobj(response, dest)
            if file_sha256(tmp_path) != IWSLT_SHA256[split]:
                raise ValueError(f"Checksum mismatch for {split}. Rerun the download cell.")
            tmp_path.replace(path)
        finally:
            tmp_path.unlink(missing_ok=True)
    translations = pq.read_table(path, columns=["translation"]).column("translation").to_pylist()
    return [(str(i), pair["de"], pair["en"]) for i, pair in enumerate(translations)]

def read_tsv(split):
    path = Path(CFG["data_dir"]) / f"{split}.tsv"
    with path.open(encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f, delimiter="\t")
        if not {"source", "target"}.issubset(reader.fieldnames or []):
            raise ValueError(f"{path} needs tab-separated source and target column headers.")
        rows = []
        for i, row in enumerate(reader):
            if None in row or row["source"] is None or row["target"] is None:
                raise ValueError(f"Malformed TSV row {i + 2} in {path}. Quote embedded tabs.")
            row_id = row.get("id")
            rows.append((str(i) if not row_id else row_id, row["source"], row["target"]))
        return rows

def smoke_splits():
    names = ["Anna", "Ben", "Clara", "David", "Emma", "Felix"]
    verbs = [("kauft", "buys"), ("sieht", "sees"), ("braucht", "needs"), ("mag", "likes")]
    foods = [("Brot", "bread"), ("Reis", "rice"), ("Milch", "milk"), ("Tee", "tea")]
    pairs = [(f"{name} {de_v} {de_n}.", f"{name} {en_v} {en_n}.")
             for name in names for de_v, en_v in verbs for de_n, en_n in foods]
    random.Random(4309).shuffle(pairs)
    parts = (pairs[:72], pairs[72:84], pairs[84:])
    return {split: [(str(i), s, t) for i, (s, t) in enumerate(part)]
            for split, part in zip(SPLIT_NAMES, parts)}

mode = CFG["data_mode"]
if mode in ("iwslt", "smoke") and (CFG["source_lang"], CFG["target_lang"]) != ("German", "English"):
    raise ValueError("The built-in datasets use German → English. Choose tsv for another pair.")
if mode == "smoke":
    raw_splits = smoke_splits()
elif mode == "iwslt":
    raw_splits = {split: read_iwslt(split) for split in SPLIT_NAMES}
elif mode == "tsv":
    raw_splits = {split: read_tsv(split) for split in SPLIT_NAMES}
else:
    raise ValueError("Choose data_mode = 'iwslt', 'tsv', or 'smoke'.")
print("Loaded rows:", {name: len(rows) for name, rows in raw_splits.items()})

### Choose fixed examples before learning the vocabulary

We normalize Unicode to NFC, collapse whitespace, and keep capitalization and punctuation. We take the first translation when a normalized source repeats within a split. Validation excludes sources already selected for training, and test excludes sources selected for either earlier split. This order lets us construct the training set independently of validation and test.

A fixed shuffle per split and the three example limits select the starting cohort. The tokenizer-length limits are applied **after** that selection, so the final counts can be smaller. Set a limit to `None` to keep all eligible examples. The notebook exports the retained examples later so your teammates can evaluate M0 and M1 on exactly the same sentences.

In [ ]:
def clean_sentence(text):
    if not isinstance(text, str):
        raise ValueError(f"Expected a text sentence, received {type(text).__name__}.")
    return " ".join(unicodedata.normalize("NFC", text).split())

def prompt_text(source):
    return CFG["prompt_template"].format(
        source_lang=CFG["source_lang"], source=source, target_lang=CFG["target_lang"]
    )

selected = {}
data_report = {"data_mode": mode, "splits": {}}
if mode == "iwslt":
    data_report.update(iwslt_revision=IWSLT_REVISION, parquet_sha256=IWSLT_SHA256)
seen_earlier = set()
for offset, split in enumerate(SPLIT_NAMES):
    candidates, seen_here, seen_ids = [], set(), set()
    stats = {"raw_rows": len(raw_splits[split]), "empty": 0,
             "repeated_source": 0, "source_in_earlier_split": 0}
    for row_id, source, target in raw_splits[split]:
        source, target = clean_sentence(source), clean_sentence(target)
        if not source or not target:
            stats["empty"] += 1
            continue
        if source in seen_earlier:
            stats["source_in_earlier_split"] += 1
            continue
        if source in seen_here:
            stats["repeated_source"] += 1
            continue
        if row_id in seen_ids:
            raise ValueError(f"Duplicate id {row_id!r} in {split}. Give each source a unique id.")
        seen_here.add(source)
        seen_ids.add(row_id)
        candidates.append({"id": row_id, "source": source, "target": target})
    random.Random(CFG["split_seed"] + offset).shuffle(candidates)
    limit = CFG[f"max_{split}"]
    if limit is not None and limit < 1:
        raise ValueError(f"max_{split} must be positive or None.")
    rows = candidates[:limit]
    if not rows:
        raise ValueError(f"No examples available in {split} after cleaning and source checks.")
    selected[split] = rows
    stats.update(eligible_before_cap=len(candidates), selected_before_length=len(rows))
    data_report["splits"][split] = stats
    seen_earlier.update(row["source"] for row in rows)

del raw_splits, candidates, seen_earlier, seen_here, seen_ids
print("Prompt an inference method will receive:")
print(repr(prompt_text(selected["train"][0]["source"])))

### Learn the tokenizer from training pairs

Byte-pair encoding (BPE) starts from 256 possible byte values. It repeatedly merges frequent neighboring symbols, letting common words or word fragments use fewer token positions. Here it learns merge rules from the selected **training** prompts and English continuations. The byte alphabet can represent new spellings and characters. No pretrained tokenizer or pretrained model weights enter M2.

The model reads a beginning-of-sequence token (BOS) and the prompt, then predicts the target, including its leading space, and an end-of-sequence token (EOS). We encode the prompt and continuation separately so the loss mask can mark their boundary exactly. PAD fills short examples when batching. The vocabulary size printed below is the actual number of token IDs that the model must embed. A small smoke corpus can produce fewer merges than requested.

In [ ]:
from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers

special_tokens = ["<|pad|>", "<|bos|>", "<|eos|>"]
if CFG["vocab_size"] < 256 + len(special_tokens):
    raise ValueError("vocab_size must accommodate the 256 bytes and three special tokens.")
tokenizer = Tokenizer(models.BPE())
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tokenizer.decoder = decoders.ByteLevel()
trainer = trainers.BpeTrainer(
    vocab_size=CFG["vocab_size"], min_frequency=2, special_tokens=special_tokens,
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(), show_progress=False,
)

def training_texts():
    for row in selected["train"]:
        yield prompt_text(row["source"])
        yield " " + row["target"]

tokenizer.train_from_iterator(training_texts(), trainer=trainer, length=2 * len(selected["train"]))
# Treat marker spellings inside user text as ordinary characters. We insert controls by ID.
tokenizer.encode_special_tokens = True
PAD_ID, BOS_ID, EOS_ID = (tokenizer.token_to_id(s) for s in special_tokens)
VOCAB_SIZE = tokenizer.get_vocab_size()
TOKENIZER_PATH = OUT / "tokenizer.json"
tokenizer.save(str(TOKENIZER_PATH))

prompt_limit = CFG["context_len"] - CFG["max_new_tokens"]
def encode_prompt(source):
    source = clean_sentence(source)
    if not source:
        raise ValueError("Enter a nonempty source sentence.")
    ids = [BOS_ID] + tokenizer.encode(prompt_text(source)).ids
    if len(ids) > prompt_limit:
        raise ValueError(f"Prompt has {len(ids)} tokens; the configured limit is {prompt_limit}.")
    return ids

def decode_target(ids):
    ids = [int(i) for i in ids]
    if EOS_ID in ids:
        ids = ids[:ids.index(EOS_ID)]
    return tokenizer.decode(ids, skip_special_tokens=True).strip()

print("Vocabulary:", VOCAB_SIZE, "| PAD, BOS, EOS:", PAD_ID, BOS_ID, EOS_ID)
print("Available prompt positions (including BOS):", prompt_limit)

### Define the training cohort and preserve complete references

Every retained source must leave room for the **same maximum generation length**. M2 trains on a pair only if its full prompt, reference, and EOS fit the context window. For validation and test, all source-eligible rows remain available for BLEU and chrF, including ones with long references. Teacher-forced cross-entropy is reported on the subset whose full reference fits. We save both counts so you can state precisely what each evaluation covers. No reference is truncated.

`shared_splits/` contains normalized, tab-separated text and stable row IDs for the other systems. `data_report.json` records exclusions, the tokenizer digest, and the exact data fingerprint. The separate `encoded` view supplies training and cross-entropy evaluation; the decoding benchmark consumes the `splits` view with full references.

In [ ]:
splits = {name: [] for name in SPLIT_NAMES}
encoded = {name: [] for name in SPLIT_NAMES}
shared_dir = OUT / "shared_splits"
shared_dir.mkdir(parents=True, exist_ok=True)
data_digest = hashlib.sha256()
data_report["tokenizer_sha256"] = file_sha256(TOKENIZER_PATH)
data_digest.update(data_report["tokenizer_sha256"].encode())
for split in SPLIT_NAMES:
    stats = data_report["splits"][split]
    stats.update(prompt_too_long=0, pair_too_long_for_ce=0)
    for row in selected[split]:
        p = [BOS_ID] + tokenizer.encode(prompt_text(row["source"])).ids
        if len(p) > prompt_limit:
            stats["prompt_too_long"] += 1
            continue
        t = tokenizer.encode(" " + row["target"]).ids + [EOS_ID]
        if len(p) + len(t) <= CFG["context_len"]:
            encoded[split].append((p, t))
        else:
            stats["pair_too_long_for_ce"] += 1
            if split == "train":
                continue
        entry = {**row, "prompt_ids": p, "target_ids": t}
        splits[split].append(entry)
        record = [split, row["id"], row["source"], row["target"], p, t]
        data_digest.update((json.dumps(record, ensure_ascii=False) + "\n").encode())
    if not encoded[split]:
        raise ValueError(f"No complete {split} pairs fit. Increase context_len, reduce "
                         "max_new_tokens, or select more examples.")
    with (shared_dir / f"{split}.tsv").open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["id", "source", "target"], delimiter="\t",
                                extrasaction="ignore")
        writer.writeheader()
        writer.writerows(splits[split])
    stats.update(retained_rows=len(splits[split]), complete_pairs_for_ce=len(encoded[split]))
    print(f"{split:10} selected={stats['selected_before_length']:6}  "
          f"retained={len(splits[split]):6}  full pairs for CE={len(encoded[split]):6}")
DATA_SIGNATURE = data_digest.hexdigest()
data_report.update(data_signature=DATA_SIGNATURE, vocabulary_size=VOCAB_SIZE,
                   context_len=CFG["context_len"], max_new_tokens=CFG["max_new_tokens"])
(OUT / "data_report.json").write_text(json.dumps(data_report, indent=2), encoding="utf-8")
example = splits["train"][0]
assert encode_prompt(example["source"]) == example["prompt_ids"]
assert decode_target(example["target_ids"]) == example["target"]
print("One training pair\n", prompt_text(example["source"]), example["target"])
print("Data and tokenizer SHA-256:", DATA_SIGNATURE)

## Build M2 from random weights

M2 is a **decoder-only Transformer**. Its input during training is a single sequence containing the source prompt followed by the English continuation. Each block first mixes information across earlier positions with causal self-attention, then transforms each position with a two-layer feed-forward network. Learned position embeddings tell the model where each token occurs. The same word-embedding matrix reads vocabulary IDs and scores possible next IDs, so it is counted once.

PyTorch supplies matrix operations, automatic differentiation and the attention kernel. The block, model, loss and training loop are defined here. All trainable weights start from random initialization or standard normalization constants. No pretrained model or embeddings are loaded. The parameter count below is the number of distinct trainable scalars, including embeddings. A larger count increases model capacity and memory use. The brief sets no target count for M2.

In [ ]:
import hashlib
import json
import math
from pathlib import Path

import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader


class TransformerBlock(nn.Module):
    def __init__(self, width, heads, ff_width, dropout):
        super().__init__()
        if width % heads:
            raise ValueError("d_model must be divisible by n_heads.")
        self.heads = heads
        self.head_width = width // heads
        self.attn_norm = nn.LayerNorm(width)
        self.qkv = nn.Linear(width, 3 * width)
        self.attn_out = nn.Linear(width, width)
        self.ff_norm = nn.LayerNorm(width)
        self.ff = nn.Sequential(
            nn.Linear(width, ff_width), nn.GELU(), nn.Linear(ff_width, width)
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        b, t, width = x.shape
        qkv = self.qkv(self.attn_norm(x)).view(b, t, 3, self.heads, self.head_width)
        q, k, v = qkv.permute(2, 0, 3, 1, 4).unbind(0)
        z = F.scaled_dot_product_attention(
            q, k, v, is_causal=True,
            dropout_p=self.dropout.p if self.training else 0.0,
        )
        z = z.transpose(1, 2).contiguous().view(b, t, width)
        x = x + self.dropout(self.attn_out(z))
        return x + self.dropout(self.ff(self.ff_norm(x)))


class TinyGPT(nn.Module):
    def __init__(self):
        super().__init__()
        width = CFG["d_model"]
        self.max_context = CFG["context_len"]
        self.token_embedding = nn.Embedding(VOCAB_SIZE, width)
        self.position_embedding = nn.Embedding(self.max_context, width)
        self.dropout = nn.Dropout(CFG["dropout"])
        self.blocks = nn.Sequential(*[
            TransformerBlock(width, CFG["n_heads"], CFG["d_ff"], CFG["dropout"])
            for _ in range(CFG["n_layers"])
        ])
        self.final_norm = nn.LayerNorm(width)
        for layer in self.modules():
            if isinstance(layer, (nn.Embedding, nn.Linear)):
                nn.init.normal_(layer.weight, mean=0.0, std=0.02)
                if isinstance(layer, nn.Linear):
                    nn.init.zeros_(layer.bias)

    def hidden(self, ids):
        if ids.size(1) > self.max_context:
            raise ValueError("Input exceeds the model's position table.")
        positions = torch.arange(ids.size(1), device=ids.device)
        x = self.token_embedding(ids) + self.position_embedding(positions)
        return self.final_norm(self.blocks(self.dropout(x)))

    def project(self, hidden):
        return F.linear(hidden, self.token_embedding.weight)

    def forward(self, ids):
        return self.project(self.hidden(ids))

    def next_logits(self, ids):
        # All prefixes in a generation batch end in the final input column.
        return self.project(self.hidden(ids)[:, -1])


probe = TinyGPT()
PARAMS = sum(p.numel() for p in probe.parameters() if p.requires_grad)
print(f"M2 trainable parameters: {PARAMS:,} | vocabulary: {VOCAB_SIZE:,}")
del probe

## Align the prompt, next-token labels and padding

Suppose the prompt is $[p_0,p_1,p_2]$ and the English continuation is $[t_0,t_1,\mathrm{EOS}]$. The model input is $[p_0,p_1,p_2,t_0,t_1]$. The five labels are $[\mathrm{ignore},\mathrm{ignore},t_0,t_1,\mathrm{EOS}]$. At position $p_2$, the model sees the complete prompt and predicts the first English token. It learns where to stop by predicting EOS. During teacher forcing, earlier English tokens in the input come from the human translation.

Batches are padded on the right. Causality prevents real tokens from attending to later padding, and ignored labels remove padding from the loss. We count **prompt tokens** including BOS and the language labels, **target tokens** including EOS, and **input tokens** actually presented at nonpadding positions. One example has $N_\mathrm{input}=N_\mathrm{prompt}+N_\mathrm{target}-1$, since its final EOS is predicted. Repeated examples count again on every epoch. These counts measure exposure and have no universal good value. Equal counts across training seeds make the experiment comparable.

The cell also prints the planned budget from the actual encoded training rows, so you can see the cost before training starts. For the unmodified German–English defaults with a 4,096-token vocabulary, a verified preprocessing run retained 19,709 pairs. Each epoch supplies 544,606 target labels and 1,202,406 nonpadding input positions. The final training table counts what the optimizer run actually consumed.

The checks verify label alignment, ignored padding and the causal direction. Changing a future token must leave earlier predictions unchanged.

In [ ]:
IGNORE = -100


def collate_targets(examples):
    # An example contains (prompt_ids including BOS, target_ids including EOS).
    width = max(len(prompt) + len(target) - 1 for prompt, target in examples)
    inputs = torch.full((len(examples), width), PAD_ID, dtype=torch.long)
    labels = torch.full_like(inputs, IGNORE)
    prompt_count = sum(len(prompt) for prompt, _ in examples)
    target_count = sum(len(target) for _, target in examples)
    for i, (prompt, target) in enumerate(examples):
        stop = len(prompt) + len(target) - 1
        inputs[i, :stop] = torch.tensor([*prompt, *target[:-1]], dtype=torch.long)
        labels[i, len(prompt) - 1:stop] = torch.tensor(target, dtype=torch.long)
    return inputs, labels, prompt_count, target_count


def make_loader(examples, shuffle_seed=None):
    if not examples:
        raise ValueError("The split has no context-fitting examples for cross-entropy.")
    generator = None
    if shuffle_seed is not None:
        generator = torch.Generator().manual_seed(shuffle_seed)
    return DataLoader(
        examples, batch_size=CFG["batch_size"], collate_fn=collate_targets,
        shuffle=shuffle_seed is not None, generator=generator, num_workers=0,
        pin_memory=(DEVICE.type == "cuda"),
    )


# Deliberately use unequal example lengths so one label is padding.
x_demo, y_demo, _, _ = collate_targets([
    ([BOS_ID, 4, 5], [6, EOS_ID]), ([BOS_ID, 4], [EOS_ID])
])
assert y_demo[0].tolist() == [IGNORE, IGNORE, 6, EOS_ID]
assert y_demo[1].tolist() == [IGNORE, EOS_ID, IGNORE, IGNORE]
probe = TinyGPT().eval()
with torch.no_grad():
    original = probe(x_demo[:1])
    modified = x_demo[:1].clone()
    modified[0, -1] = EOS_ID
    torch.testing.assert_close(probe(modified)[:, :-1], original[:, :-1])
    torch.testing.assert_close(probe.next_logits(x_demo[:1]), original[:, -1])
del probe
print("Label alignment, padding and causal attention checks passed.")
targets_per_epoch = sum(len(t) for p, t in encoded["train"])
positions_per_epoch = sum(len(p) + len(t) - 1 for p, t in encoded["train"])
print(f"Planned per seed for {CFG['epochs']} epochs: "
      f"{CFG['epochs'] * targets_per_epoch:,} supervised target tokens and "
      f"{CFG['epochs'] * positions_per_epoch:,} nonpadding input positions.")

## Optimize target-token cross-entropy

At each supervised position the model assigns probability $p_\theta(y_j\mid\mathrm{prompt},y_{<j})$ to the correct next token. The loss for one update is the **mean negative log probability** of its supervised tokens. AdamW updates the weights with the configured learning rate and weight decay. Gradient clipping bounds the gradient norm by the configured threshold before the update. Validation and test passes turn off dropout and weight updates.

The training and held-out **cross-entropy** reported below is total negative log probability divided by the **total target-token count**, in nats per token. Longer sentences contribute more terms. Lower is better, with zero as the ideal and $\log V$ for a model that gives all $V$ tokens equal probability. Lower training cross-entropy shows that the model fits its training data. Lower validation cross-entropy governs checkpoint selection. A widening train/validation gap can indicate overfitting.

**Perplexity** is $\exp(\mathrm{CE})$, or the inverse geometric-mean probability assigned to the correct token. For example, a cross-entropy of $\log 4$ means perplexity 4. Lower is better, with 1 as the ideal. Absolute values depend on the tokenizer, corpus and token inclusion rules. Compare M2 checkpoints under the same tokenizer. BLEU and chrF support text-level comparisons with M1. Held-out cross-entropy and perplexity describe the checkpoint independently of the decoder.

In [ ]:
USE_AMP = DEVICE.type == "cuda"
AMP_DTYPE = (
    torch.bfloat16
    if USE_AMP and torch.cuda.is_bf16_supported()
    else torch.float16
)


def run_pass(net, examples, optimizer=None, scaler=None, shuffle_seed=None):
    training = optimizer is not None
    net.train(training)
    nll_sum = torch.zeros((), dtype=torch.float64, device=DEVICE)
    prompt_count = target_count = 0
    for inputs, labels, n_prompt, n_target in make_loader(examples, shuffle_seed):
        inputs = inputs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        with torch.set_grad_enabled(training):
            if training:
                optimizer.zero_grad(set_to_none=True)
            with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
                # Expose the full prompt as context, project only supervised hidden states.
                keep = labels.ne(IGNORE)
                logits = net.project(net.hidden(inputs)[keep])
                batch_nll = F.cross_entropy(logits.float(), labels[keep], reduction="sum")
            if training:
                scaler.scale(batch_nll / n_target).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(net.parameters(), CFG["grad_clip"])
                scaler.step(optimizer)
                scaler.update()
        nll_sum += batch_nll.detach().double()
        prompt_count += n_prompt
        target_count += n_target
    ce = nll_sum.item() / target_count
    if not math.isfinite(ce):
        raise FloatingPointError("Nonfinite cross-entropy. Check the learning rate and AMP settings.")
    return {
        "ce": ce, "ppl": math.exp(ce), "examples": len(examples),
        "prompt_tokens": prompt_count, "target_tokens": target_count,
        "input_tokens": prompt_count + target_count - len(examples),
    }


@torch.no_grad()
def target_metrics(net, examples):
    return run_pass(net, examples)

## Train each seed and choose its best epoch on validation

All seeds see the same examples and complete the same number of epochs. A seed changes initialization, minibatch order and dropout. An **epoch** is one full traversal of the retained training examples. The log shows each epoch's average training loss and validation loss, then identifies the epoch with lowest validation cross-entropy. Training loss averages predictions made while weights are changing. Validation evaluates a single checkpoint.

An epoch checkpoint contains the current weights, optimizer, mixed-precision scaler, training history and best weights. The checkpoint is replaced only after a full epoch. If a session stops mid-epoch, rerunning the notebook resumes from the previous completed epoch when the same output files are available. Data, architecture, training settings, software and device identity determine the run directory. To resume after a Colab runtime reset, use a persistent output directory from the configuration section.

In [ ]:
MODEL_KEYS = (
    "d_model", "n_heads", "n_layers", "d_ff", "dropout", "context_len",
    "batch_size", "epochs", "lr", "weight_decay", "grad_clip",
)
model_settings = {
    "model_revision": 1, "data_signature": DATA_SIGNATURE, "vocab_size": VOCAB_SIZE,
    "torch_version": str(torch.__version__), "amp": str(AMP_DTYPE) if USE_AMP else "off",
    "device": torch.cuda.get_device_name(DEVICE) if USE_AMP else str(DEVICE),
    **{key: CFG[key] for key in MODEL_KEYS},
}
identity = hashlib.sha256(json.dumps(model_settings, sort_keys=True).encode()).hexdigest()[:16]
MODEL_DIR = OUT / f"m2_{identity}"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
(MODEL_DIR / "settings.json").write_text(json.dumps(model_settings, indent=2))


def model_path(seed):
    return MODEL_DIR / f"seed_{seed}.pt"


def load_best_model(seed):
    state = torch.load(model_path(seed), map_location="cpu", weights_only=True)
    net = TinyGPT()
    net.load_state_dict(state["best_model"])
    return net.to(DEVICE).eval()


def train_seed(seed):
    torch.manual_seed(seed)
    net = TinyGPT().to(DEVICE)
    optimizer = torch.optim.AdamW(
        net.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"]
    )
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP and AMP_DTYPE == torch.float16)
    history, best_model, best_epoch = [], None, None
    best_val = float("inf")
    path = model_path(seed)
    if path.exists():
        state = torch.load(path, map_location="cpu", weights_only=True)
        if state["identity"] != identity or state["seed"] != seed:
            raise ValueError(f"Checkpoint settings differ at {path}")
        net.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        scaler.load_state_dict(state["scaler"])
        history, best_model = state["history"], state["best_model"]
        best_val, best_epoch = state["best_val"], state["best_epoch"]
        del state
        print(f"Seed {seed}: loaded {len(history)} completed epochs.")

    for epoch in range(len(history) + 1, CFG["epochs"] + 1):
        # Reset the per-epoch streams so a restarted epoch gets the same shuffle/dropout seed.
        epoch_seed = seed + 1_000_003 * epoch
        torch.manual_seed(epoch_seed)
        train = run_pass(net, encoded["train"], optimizer, scaler, epoch_seed)
        val = target_metrics(net, encoded["validation"])
        row = {"seed": seed, "epoch": epoch}
        row.update({f"train_{key}": value for key, value in train.items()})
        row.update({f"val_{key}": value for key, value in val.items()})
        history.append(row)
        if val["ce"] < best_val:
            best_val, best_epoch = val["ce"], epoch
            best_model = {k: v.detach().to("cpu", copy=True) for k, v in net.state_dict().items()}
        snapshot = {
            "identity": identity, "seed": seed, "model": net.state_dict(),
            "optimizer": optimizer.state_dict(), "scaler": scaler.state_dict(),
            "history": history, "best_model": best_model,
            "best_epoch": best_epoch, "best_val": best_val,
        }
        temporary = path.with_suffix(".tmp")
        torch.save(snapshot, temporary)
        temporary.replace(path)
        print(f"Seed {seed} | epoch {epoch}/{CFG['epochs']} | "
              f"train CE {train['ce']:.3f} | val CE {val['ce']:.3f} | "
              f"target tokens this epoch {train['target_tokens']:,}")
    return history, best_epoch


if CFG["epochs"] < 1 or not CFG["train_seeds"] or len(set(CFG["train_seeds"])) != len(CFG["train_seeds"]):
    raise ValueError("Use at least one epoch and a nonempty list of distinct training seeds.")
histories = {}
selected_epochs = {}
for seed in CFG["train_seeds"]:
    histories[seed], selected_epochs[seed] = train_seed(seed)
training_history = pd.DataFrame([row for rows in histories.values() for row in rows])
training_history.to_csv(MODEL_DIR / "training_history.csv", index=False)

## Report M2 parameters, actual training budgets and held-out performance

Each row corresponds to an independently initialized M2. **Full** counts include all completed epochs, even those after the selected checkpoint. **To best** counts include just the epochs that produced the selected weights. The table distinguishes prompt exposure, causal input positions and English target labels, so the report can state exactly which token budget it uses. Summing the full counts across seeds gives total project training exposure. Validation and test tokens are excluded from every training budget.

The frozen checkpoints are scored on all context-fitting test references after epoch selection. **Test CE coverage** gives the number of scored references and the number of source-eligible test items. Long references may still participate in translation evaluation even if teacher-forced scoring would exceed context. If multiple models are trained, the sample standard deviation measures how much the scores vary across seeds. Smaller values indicate more stable results. The first listed training seed is the prespecified checkpoint for the decoding study. Each decoder therefore translates the same sentences with exactly the same weights. Variability from repeated decoding seeds is a separate quantity from training-seed variability.

In [ ]:
summary_rows = []
for seed in CFG["train_seeds"]:
    history = pd.DataFrame(histories[seed])
    best_epoch = selected_epochs[seed]
    selected = history.loc[history.epoch == best_epoch].iloc[0]
    frozen = load_best_model(seed)
    test = target_metrics(frozen, encoded["test"])
    row = {
        "seed": seed, "trainable_parameters": PARAMS, "completed_epochs": len(history),
        "best_epoch": best_epoch, "best_val_ce": float(selected.val_ce),
        "best_val_ppl": float(selected.val_ppl), "test_ce": test["ce"],
        "test_ppl": test["ppl"], "test_target_tokens": test["target_tokens"],
        "test_ce_rows": test["examples"], "test_source_eligible_rows": len(splits["test"]),
    }
    for budget in ("prompt_tokens", "target_tokens", "input_tokens"):
        column = f"train_{budget}"
        row[f"full_{budget}"] = int(history[column].sum())
        row[f"to_best_{budget}"] = int(history.loc[history.epoch <= best_epoch, column].sum())
    summary_rows.append(row)
    del frozen
training_summary = pd.DataFrame(summary_rows)
training_summary.to_csv(MODEL_DIR / "training_summary.csv", index=False)
print(f"Actual trainable parameters per M2: {PARAMS:,}")
print(f"Full supervised training tokens across all seeds: {training_summary.full_target_tokens.sum():,}")
display(training_summary)
if len(training_summary) > 1:
    print("Held-out cross-entropy and perplexity across training seeds (sample SD):")
    display(training_summary[["best_val_ce", "test_ce", "test_ppl"]].agg(["mean", "std"]))
model = load_best_model(CFG["train_seeds"][0])

## Decode a translation from the trained M2

The model sees the source prompt and predicts one target token at a time. A greedy translation always takes the highest-probability token. An ancestral sample draws a token from the whole allowed probability distribution at temperature 1, appends it, and repeats. The only masked tokens are PAD and BOS. EOS can be drawn at the first step. If the length limit arrives first, the candidate is marked as capped.

For one source, the model scores all active candidates together. A candidate that emits EOS leaves the batch. Each sample has its own random-number stream derived from the decoding seed and its candidate index. This lets a larger pool reuse the same random draws for its first candidates. Small floating-point differences across batch sizes are possible on a GPU.

Three counters help explain the measured latency. **Forward calls** counts model invocations. **Forward rows** counts individual candidate prefixes scored across those invocations. **Prefix tokens** sums the lengths of those prefixes, measuring input positions processed by this implementation, which recomputes the prefix each step. A fourth counter, **generated tokens**, counts tokens in the returned candidates, including an emitted EOS. For beam search this last counter describes the winning translation only. Forward rows and prefix tokens also account for its discarded alternatives.

In [ ]:
import numpy as np
import torch
from sacrebleu.metrics import CHRF

def _next_token_logits(model, prefixes, work):
    rows, length = prefixes.shape
    work["forward_calls"] += 1
    work["forward_rows"] += rows
    work["prefix_tokens"] += rows * length
    logits = model.next_logits(prefixes).float().clone()
    logits[:, [PAD_ID, BOS_ID]] = -torch.inf
    return logits

def _draw_paths(model, prompt, count, cap, sample, seed, device, work):
    paths = [[] for _ in range(count)]
    active = list(range(count))
    prefixes = torch.tensor(prompt, dtype=torch.long, device=device).expand(count, -1)
    if sample:
        # One uniform variate per candidate and step for inverse-CDF sampling.
        streams = [np.random.default_rng([seed, j]).random(cap) for j in active]
        uniforms = torch.as_tensor(np.stack(streams), device=device)

    for step in range(cap):
        logits = _next_token_logits(model, prefixes, work)
        if sample:
            cdf = logits.softmax(dim=-1).cumsum(dim=-1, dtype=torch.float64)
            draws = uniforms[active, step] * cdf[:, -1]
            next_ids = torch.searchsorted(cdf, draws[:, None], right=True).squeeze(1)
            next_ids = next_ids.clamp_max(cdf.shape[1] - 1)
        else:
            next_ids = logits.argmax(dim=-1)

        survivors = []
        for row, token_id in enumerate(next_ids.tolist()):
            paths[active[row]].append(token_id)
            if token_id != EOS_ID:
                survivors.append(row)
        if not survivors or step + 1 == cap:
            break
        prefixes = torch.cat((prefixes, next_ids[:, None]), dim=1)[survivors]
        active = [active[row] for row in survivors]
    return paths

### Beam search

Beam search keeps several possible translations alive. At each step it considers every allowed next token from every live prefix, together with the already finished hypotheses still in the beam. It retains the best `width` hypotheses overall. A finished hypothesis competes at its original score and never receives another token. At the common generation cap, the best retained hypothesis wins, even if it has yet to emit EOS.

We rank a hypothesis with target length $L$ and cumulative log probability $s$ by $s/((5+L)/6)^\alpha$. The length counts EOS when emitted. Higher scores are better. Setting $\alpha=0$ uses the raw log probability. Positive $\alpha$ reduces the preference for short translations. The experiment fixes this parameter before comparing methods. A width of one gives greedy decoding.

In [ ]:
def _beam_path(model, prompt, width, cap, alpha, device, work):
    # Each entry stores (generated IDs, cumulative log probability).
    live = [([], 0.0)]
    finished = []
    best_path = []
    for length in range(1, cap + 1):
        batch = torch.tensor([prompt + path for path, _ in live],
                             dtype=torch.long, device=device)
        log_probs = _next_token_logits(model, batch, work).log_softmax(dim=-1)
        live_scores = log_probs.new_tensor([score for _, score in live])
        expanded = (log_probs + live_scores[:, None]).flatten()
        vocab_size = log_probs.shape[1]
        penalty = ((5 + length) / 6) ** alpha
        # Finished paths keep the length penalty for the step when they ended.
        old_raw = log_probs.new_tensor([score for _, score in finished])
        old_rank = log_probs.new_tensor([
            score / (((5 + len(path)) / 6) ** alpha) for path, score in finished
        ])
        ranks = torch.cat((expanded / penalty, old_rank))
        raw = torch.cat((expanded, old_raw))
        indices = ranks.topk(min(width, ranks.numel())).indices
        survivors = []
        for index, score in zip(indices.tolist(), raw[indices].tolist()):
            if not np.isfinite(score):
                continue
            if index < expanded.numel():
                parent, token_id = divmod(index, vocab_size)
                path = live[parent][0] + [token_id]
            else:
                path = finished[index - expanded.numel()][0]
            survivors.append((path, score))
        if not survivors:
            raise RuntimeError("No finite beam hypothesis; check model logits.")
        best_path = survivors[0][0]
        live = [item for item in survivors if item[0][-1] != EOS_ID]
        finished = [item for item in survivors if item[0][-1] == EOS_ID]
        if not live:
            break
    return [best_path]

@torch.inference_mode()
def generate_candidates(model, prompt_ids, method, *, width=1, max_new_tokens,
                        seed=0, length_penalty=0.6):
    # Deliberately accepts only the source prompt; the reference stays in evaluation.
    prompt = list(prompt_ids)
    if method not in {"greedy", "beam", "sample"}:
        raise ValueError("method must be greedy, beam, or sample.")
    if width < 1 or max_new_tokens < 1 or length_penalty < 0 or seed < 0:
        raise ValueError("Use positive width/cap and nonnegative penalty/seed.")
    if method == "greedy" and width != 1:
        raise ValueError("Greedy decoding has width 1.")
    if not prompt or prompt[0] != BOS_ID or len(prompt) + max_new_tokens > model.max_context:
        raise ValueError("The prompt needs BOS and enough context space for the entire cap.")
    model.eval()
    device = next(model.parameters()).device
    work = dict(forward_calls=0, forward_rows=0, prefix_tokens=0)
    if method == "beam" and width > 1:
        paths = _beam_path(model, prompt, width, max_new_tokens, length_penalty, device, work)
    else:
        paths = _draw_paths(model, prompt, width, max_new_tokens, method == "sample",
                            seed, device, work)
    candidates = [
        {"text": decode_target(path), "token_ids": path, "ended_eos": path[-1] == EOS_ID}
        for path in paths
    ]
    work["generated_tokens"] = sum(map(len, paths))
    return candidates, work

### Minimum Bayes risk with sentence chrF

Suppose M2 samples three translations. Two may phrase the same idea similarly while the third gives a different answer. For each possible output, we treat each *other sample* in turn as a pseudo-reference and average sentence chrF. We return the sampled output with the highest average. This is equivalent to minimizing estimated risk with loss $100-\mathrm{chrF}$. chrF compares character $n$-grams and its default $\beta=2$ emphasizes recall. The first argument to chrF is the output being considered. Argument order matters.

For $k>1$, the score of sample $i$ is $\frac{1}{k-1}\sum_{j\ne i}\mathrm{chrF}(y_i,y_j)$. Excluding the same index avoids rewarding a candidate for comparing with itself. Two independent draws that produce identical text still count separately. This implementation makes exactly $k(k-1)$ ordered utility comparisons: 12 at $k=4$ and 56 at $k=8$. At $k=1$ we return the sole draw without calculating an agreement score. Ties go to the first sampled candidate. The highest agreement can still select a bad translation if the model repeats the same mistake. Only the later corpus BLEU and chrF measurements use human references. [Background on sampling-based MBR for translation](https://aclanthology.org/2022.emnlp-main.754/)

In [ ]:
MBR_CHRF = CHRF(char_order=6, word_order=0, beta=2)

def mbr_select(candidates):
    k = len(candidates)
    if k == 0:
        raise ValueError("MBR needs at least one sample.")
    if k == 1:
        return 0, None, 0
    scores = []
    for i, candidate in enumerate(candidates):
        utility = sum(
            MBR_CHRF.sentence_score(candidate["text"], [other["text"]]).score
            for j, other in enumerate(candidates) if j != i
        )
        scores.append(utility / (k - 1))
    best = max(range(k), key=scores.__getitem__)
    return best, scores[best], k * (k - 1)

### Check decoding on a tiny probability tree

This check makes a case whose answer we can calculate by hand. The first token A has probability 0.60 and B has probability 0.40. Greedy takes A and then A, for a capped path with probability $0.60\times0.51=0.306$. B always leads to EOS, so a width-two beam can find the more probable complete path with probability $0.40$. Separate checks cover agreement voting, the $k=1$ case, and repeatable sampling.

In [ ]:
def check_decoders():
    a, b = [i for i in range(VOCAB_SIZE) if i not in {PAD_ID, BOS_ID, EOS_ID}][:2]

    class ToyDecoder(torch.nn.Module):
        max_context = 8
        def __init__(self):
            super().__init__()
            size = max(a, b, BOS_ID, EOS_ID, PAD_ID) + 1
            table = torch.full((size, size), -torch.inf)
            table[BOS_ID, [a, b]] = torch.tensor([0.60, 0.40]).log()
            table[a, [a, b]] = torch.tensor([0.51, 0.49]).log()
            table[b, EOS_ID] = 0.0
            self.table = torch.nn.Parameter(table, requires_grad=False)
        def next_logits(self, ids):
            return self.table[ids[:, -1]]

    toy = ToyDecoder()
    greedy, _ = generate_candidates(toy, [BOS_ID], "greedy", max_new_tokens=2)
    beam, work = generate_candidates(toy, [BOS_ID], "beam", width=2,
                                     max_new_tokens=2, length_penalty=0)
    assert greedy[0]["token_ids"] == [a, a] and not greedy[0]["ended_eos"]
    assert beam[0]["token_ids"] == [b, EOS_ID] and beam[0]["ended_eos"]
    assert work["forward_calls"] == 2 and work["prefix_tokens"] == 1 + 2 * 2
    draws, _ = generate_candidates(toy, [BOS_ID], "sample", width=2, max_new_tokens=2, seed=9)
    more, _ = generate_candidates(toy, [BOS_ID], "sample", width=4, max_new_tokens=2, seed=9)
    assert draws == more[:2]
    votes = [{"text": t} for t in ("good night", "good night", "train station")]
    winner, agreement, comparisons = mbr_select(votes)
    assert winner == 0 and 50 <= agreement <= 100 and comparisons == 6
    assert mbr_select(votes[:1]) == (0, None, 0)

check_decoders()
print("Greedy, beam search, sampling, and chrF-MBR checks passed.")

## Benchmark the decoding methods on identical held-out sources

We draw one fixed subset from the source-eligible test split and save its IDs. The same IDs, checkpoint, generation cap, and hardware are used for every comparison. Each method processes one source at a time. Its active beams or samples can be batched together. With no KV cache, the decoder recomputes prefixes at each step. This is the execution regime to state when reporting speed.

Every method gets a short untimed warm-up on a validation prompt. For each repeat we shuffle method order, then run every method independently from start to finish. Even when random draws are shared across candidate counts, a larger pool is generated and timed afresh. We repeat deterministic methods to measure timing spread. Repeating greedy on the same sentences does not give additional independent translation evidence.

**Timing boundary.** Latency starts with a tokenized source and ends with selected output text. It includes model inference, candidate detokenization, and, for MBR, CPU chrF reranking. It excludes common source preprocessing, file writes and scoring against human references. CUDA synchronization makes the timers wait for GPU work. The table reports both component times so you can see whether the model or reranker drives an increase in cost.

In [ ]:
from time import perf_counter

def synchronize_device():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize(DEVICE)

def sentence_seed(trial_seed, row_id):
    key = json.dumps([int(trial_seed), str(row_id)], ensure_ascii=False).encode()
    return int.from_bytes(hashlib.sha256(key).digest()[:8], byteorder="big")

def decode_one(source_model, prompt_ids, method, width, sample_seed, cap):
    generate_method = "sample" if method == "mbr" else method
    synchronize_device()
    start = perf_counter()
    pool, work = generate_candidates(
        source_model, prompt_ids, generate_method, width=width, max_new_tokens=cap,
        seed=sample_seed, length_penalty=CFG["length_penalty"]
    )
    synchronize_device()
    generated = perf_counter()
    if method == "mbr" and width > 1:
        best, agreement, pairs = mbr_select(pool)
        finished = perf_counter()
    else:
        best, agreement, pairs = 0, None, 0
        finished = generated
    return pool, best, {
        **work, "utility_comparisons": pairs, "agreement_chrf": agreement,
        "generation_s": generated - start, "rerank_s": finished - generated,
        "total_s": finished - start,
    }

n_eval = min(CFG["eval_sources"], len(splits["test"]))
eval_rows = random.Random(CFG["eval_subset_seed"]).sample(splits["test"], n_eval)
pd.DataFrame([{k: r[k] for k in ("id", "source", "target")} for r in eval_rows]).to_csv(
    OUT / "q3_test_subset.tsv", sep="\t", index=False)
methods = [("greedy", 1), ("beam", CFG["beam_width"])] + [
    ("mbr", k) for k in sorted(CFG["k_values"])
]
for method, width in methods:
    decode_one(model, splits["validation"][0]["prompt_ids"], method, width, 733,
               cap=min(4, CFG["max_new_tokens"]))
print(f"Frozen M2 training seed {CFG['train_seeds'][0]} | {n_eval} shared test sources | "
      f"{len(methods)} methods/settings × {len(CFG['decode_seeds'])} repeats")

### Measure quality and cost for each repeat

For each source, sampling and reranking see only the prompt. Once the translations for a method are final, we compare them with the full human references. **Corpus BLEU** measures modified word $n$-gram precision with a brevity penalty. **Corpus chrF** measures character $n$-gram precision and recall. Higher is better on the same test set, and 100 is the upper bound for both. On a small subset, BLEU can reach zero even when some words are correct. There is no universal passing score. The **sacreBLEU signatures** record exactly which metric settings produced the values. [sacreBLEU documentation](https://github.com/mjpost/sacrebleu)

Each `decoding_runs.csv` row represents one full repeat. **EOS percentage** is the share of selected outputs that stopped naturally. **Milliseconds per source** is elapsed seconds summed across sentences, divided by their number and multiplied by 1,000. Lower is faster. **Prefix tokens**, **forward rows** and **utility comparisons** count computational work. Smaller counts are useful when quality is comparable. Per-source records preserve outputs, timings and the sampled candidates for auditing. Each completed method is written to disk.

In [ ]:
quality_bleu = BLEU(tokenize="13a", lowercase=False, effective_order=False)
quality_chrf = CHRF(char_order=6, word_order=0, beta=2)
run_rows, prediction_rows = [], []
for trial_seed in CFG["decode_seeds"]:
    trial_methods = methods.copy()
    random.Random(trial_seed + 7117).shuffle(trial_methods)
    for method, width in trial_methods:
        trial_predictions = []
        for row in eval_rows:
            sample_seed = sentence_seed(trial_seed, row["id"])
            pool, best, info = decode_one(model, row["prompt_ids"], method, width,
                                          sample_seed, CFG["max_new_tokens"])
            winner = pool[best]
            trial_predictions.append({
                "id": row["id"], "source": row["source"], "reference": row["target"],
                "prediction": winner["text"], "ended_eos": winner["ended_eos"],
                "method": method, "width": width, "decode_seed": trial_seed,
                "selected_index": best, "candidates": [x["text"] for x in pool], **info,
            })
        trial = pd.DataFrame(trial_predictions)
        hypotheses, references = trial["prediction"].tolist(), trial["reference"].tolist()
        total_ms = 1_000 * trial.total_s.mean()
        result = {
            "method": method, "width": width, "decode_seed": trial_seed, "n_sources": len(trial),
            "bleu": quality_bleu.corpus_score(hypotheses, [references]).score,
            "chrf": quality_chrf.corpus_score(hypotheses, [references]).score,
            "bleu_signature": str(quality_bleu.get_signature()),
            "chrf_signature": str(quality_chrf.get_signature()),
            "ms_per_source": total_ms, "gen_ms_per_source": 1_000 * trial.generation_s.mean(),
            "rerank_ms_per_source": 1_000 * trial.rerank_s.mean(), "eos_pct": 100 * trial.ended_eos.mean(),
        }
        for name in ("prefix_tokens", "forward_calls", "forward_rows", "generated_tokens", "utility_comparisons"):
            result[f"{name}_per_source"] = trial[name].mean()
        prediction_rows.extend(trial_predictions)
        run_rows.append(result)
        pd.DataFrame(run_rows).to_csv(OUT / "decoding_runs.csv", index=False)
        pd.DataFrame(prediction_rows).to_json(OUT / "decoding_predictions.jsonl",
                                               orient="records", lines=True, force_ascii=False)
        print(f"Seed {trial_seed} | {method:6s} {width:2d} | BLEU {result['bleu']:5.2f} | "
              f"chrF {result['chrf']:5.2f} | {total_ms:7.2f} ms/source")
bench = pd.DataFrame(run_rows)
predictions = pd.DataFrame(prediction_rows)
print("BLEU:", bench.bleu_signature.iloc[0])
print("chrF:", bench.chrf_signature.iloc[0])

## Read the quality–cost curve

Each row below summarizes one decoding method on the same held-out source sentences. **Corpus BLEU** rewards word n-gram overlap and penalizes excessively short translations. **Corpus chrF** combines character n-gram precision and recall, weighting recall more heavily with $\beta=2$. Both compare the final translations with the human references on a 0–100 scale. Higher is better on this fixed test set. chrF used inside MBR compares *samples with each other* and can be high even when their translation is wrong.

**Milliseconds per source** measures end-to-end latency. Its generation and reranking components show where time went. Lower latency is better at a given quality. **Prefix tokens per source** counts the input token positions processed across forward passes in this implementation, including repeated prefix computation. It helps explain why two methods with the same output length can cost different amounts. **EOS percentage** measures how often the selected translation stopped naturally. Low values are a reason to inspect the length limit and outputs.

The table averages over decode seeds. The plot shows one point per method or candidate count. Move right to see the additional time spent and up to see improvements. The vertical bars show one sample standard deviation of corpus scores across decode seeds. Greedy and beam produce the same translations every repeat under deterministic execution, though their measured times can vary. One seed gives no standard deviation, and these bars do not estimate uncertainty over new source sentences or new trained checkpoints.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

def decoder_label(method, width):
    if method == "mbr":
        return f"MBR k={int(width)}"
    if method == "beam":
        return f"Beam width={int(width)}"
    return "Greedy"

score_columns = ["bleu", "chrf", "ms_per_source", "gen_ms_per_source",
                 "rerank_ms_per_source", "prefix_tokens_per_source", "eos_pct"]
summary = bench.groupby(["method", "width"])[score_columns].agg(["mean", "std"])
summary.columns = [f"{metric}_{stat}" for metric, stat in summary.columns]
summary["decode_runs"] = bench.groupby(["method", "width"]).size()
summary = summary.reset_index()
summary["decoder"] = [decoder_label(m, w) for m, w in zip(summary.method, summary.width)]
summary.to_csv(OUT / "decoding_summary.csv", index=False)
display(summary[["decoder", "decode_runs", "bleu_mean", "bleu_std", "chrf_mean", "chrf_std",
                 "ms_per_source_mean", "ms_per_source_std", "gen_ms_per_source_mean",
                 "rerank_ms_per_source_mean", "prefix_tokens_per_source_mean",
                 "eos_pct_mean"]].round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), constrained_layout=True)
styled_methods = [("greedy", "o", "tab:blue"), ("beam", "s", "tab:orange"),
                  ("mbr", "D", "tab:green")]
for ax, metric, title in zip(axes, ["chrf", "bleu"], ["Corpus chrF", "Corpus BLEU"]):
    for method, marker, color in styled_methods:
        points = summary[summary.method == method].sort_values("width")
        if points.empty:
            continue
        x, y = points["ms_per_source_mean"], points[f"{metric}_mean"]
        sd = points[f"{metric}_std"]
        label = "Sample + MBR" if method == "mbr" else points.iloc[0]["decoder"]
        ax.errorbar(x, y, yerr=sd.fillna(0) if sd.notna().any() else None,
                    fmt=marker, linestyle="-" if method == "mbr" else "none",
                    color=color, capsize=3, label=label)
        if method == "mbr":
            for x_i, y_i, k in zip(x, y, points.width):
                ax.annotate(f"k={k}", (x_i, y_i), xytext=(3, 6),
                            textcoords="offset points", fontsize=8)
    ax.set(xlabel="Mean end-to-end time (ms per source)", ylabel=title)
    ax.margins(x=0.12, y=0.18)
    ax.grid(alpha=0.2)
    ax.legend(fontsize=8)
fig.savefig(OUT / "quality_vs_cost.png", dpi=160, bbox_inches="tight")
plt.show()

### Check the prediction recorded before training

Compare the largest candidate pool with a single random sample, greedy decoding, and beam search. Each difference pairs runs on the same evaluation sentences and matches the decode-seed label. **ΔchrF** and **ΔBLEU** are differences in corpus-score points. A positive value favors the larger pool. The **time ratio** divides the larger pool's latency by the comparison method's latency within each repeat. A ratio of 2 means twice the time per source. The reported mean ratio weights each repeat equally.

The quantitative prediction concerns the comparison with $k=1$. It requires both an average gain of at least 1.0 chrF point and a mean time ratio above 1.0. Inspect the individual runs as well as the mean. A toy smoke run only checks the machinery.

In [ ]:
k_largest = max(CFG["k_values"])
large = bench.loc[(bench.method == "mbr") & (bench.width == k_largest)]
comparators = [("mbr", 1), ("greedy", 1), ("beam", CFG["beam_width"])]
pair_frames = []
for method, width in comparators:
    if method == "mbr" and k_largest == width:
        continue
    other = bench.loc[(bench.method == method) & (bench.width == width)]
    matched = large.merge(other, on="decode_seed", suffixes=("_large", "_base"),
                          validate="one_to_one")
    if matched.empty:
        continue
    pair_frames.append(pd.DataFrame({
        "candidate": f"MBR k={k_largest}", "comparator": decoder_label(method, width),
        "decode_seed": matched.decode_seed, "delta_chrf": matched.chrf_large - matched.chrf_base,
        "delta_bleu": matched.bleu_large - matched.bleu_base,
        "time_ratio": matched.ms_per_source_large / matched.ms_per_source_base,
    }))

paired = pd.concat(pair_frames, ignore_index=True)
paired.to_csv(OUT / "paired_comparisons.csv", index=False)
display(paired.groupby("comparator")[["delta_chrf", "delta_bleu", "time_ratio"]]
        .agg(["mean", "std"]).round(3))
k1 = paired[paired.comparator == decoder_label("mbr", 1)]
if k1.empty:
    print("The prediction needs k=1 and at least one larger candidate pool.")
else:
    gain, ratio = k1.delta_chrf.mean(), k1.time_ratio.mean()
    met = gain >= 1.0 and ratio > 1.0
    print(f"MBR k={k_largest} vs k=1: {gain:+.2f} mean chrF points; "
          f"{ratio:.2f}× mean time ratio. Both thresholds met: {met}.")

### Inspect the same translations across methods

The first six sources are a seeded random sample of the evaluation IDs (or all IDs for a smaller set). Up to three additional sources have a diagnostic flag under at least one method. Compare greedy, beam, and the largest MBR pool side by side for one decode seed. The flags help find cases worth reading. A changed digit string, for instance, could be an error or a valid formatting difference. Read the source and reference, then fill in the two manual columns in the saved CSV. Possible labels include *copied source*, *omission*, *addition*, *wrong name or number*, *runaway generation*, and *acceptable paraphrase*. Keep the random sample distinct from examples deliberately chosen for inspection.

In [ ]:
import re

review_seed = CFG["decode_seeds"][0]
chosen_runs = predictions[(predictions.decode_seed == review_seed) &
                          ((predictions.method != "mbr") | (predictions.width == k_largest))].copy()

def review_flags(row):
    source, ref, hyp = (str(row[name]).strip() for name in ["source", "reference", "prediction"])
    flags = []
    if not hyp:
        flags.append("empty output")
    if hyp and source.casefold() == hyp.casefold():
        flags.append("exact source copy")
    if sorted(re.findall(r"\d+", ref)) != sorted(re.findall(r"\d+", hyp)):
        flags.append("different digit strings")
    if not row["ended_eos"]:
        flags.append("hit length cap")
    return ", ".join(flags)

chosen_runs["flags_to_check"] = chosen_runs.apply(review_flags, axis=1)
all_ids = chosen_runs["id"].drop_duplicates()
random_ids = all_ids.sample(n=min(6, len(all_ids)), random_state=710).tolist()
flagged_ids = chosen_runs.loc[chosen_runs.flags_to_check.ne("") &
                              ~chosen_runs["id"].isin(random_ids), "id"].drop_duplicates()
extra_ids = flagged_ids.sample(n=min(3, len(flagged_ids)), random_state=711).tolist()
review_ids = random_ids + extra_ids
review = chosen_runs.loc[chosen_runs["id"].isin(review_ids)].copy()
review["selection"] = np.where(review["id"].isin(random_ids), "random", "flagged extra")
review["source_order"] = review["id"].map({key: i for i, key in enumerate(review_ids)})
review["decoder"] = [decoder_label(m, w) for m, w in zip(review.method, review.width)]
review["manual_error_type"] = ""
review["manual_notes"] = ""
review = review.sort_values(["source_order", "decoder"])
review_columns = ["selection", "id", "decode_seed", "decoder", "source", "reference", "prediction",
                  "flags_to_check", "manual_error_type", "manual_notes"]
review[review_columns].to_csv(OUT / "error_review.csv", index=False)
with pd.option_context("display.max_colwidth", 100):
    display(review.loc[review.source_order < 3, review_columns[:8]])

## Translate a new sentence with M2

Change the source and run this cell again. The table shows how one frozen M2 checkpoint translates that sentence under each decoder. The beam uses the preset width and MBR uses the largest configured pool. For MBR, the candidate list lets you see which sampled translation won. No reference sentence is needed.

The German example below is for the default project. With a different team language, replace it with a sentence in that language. The demo initially uses a validation source in that case. Inputs longer than the reserved prompt budget raise an error showing the limit.

In [ ]:
NEW_SOURCE = ("Anna kauft Brot." if PROFILE == "smoke" else
              "Wo ist der nächste Bahnhof?" if CFG["source_lang"] == "German" else
              splits["validation"][0]["source"])  # Replace with your own sentence.
demo_prompt = encode_prompt(NEW_SOURCE)
demo_rows = []
for demo_method, demo_width in [("greedy", 1), ("beam", CFG["beam_width"]),
                                ("mbr", max(CFG["k_values"]))]:
    demo_pool, chosen, demo_info = decode_one(model, demo_prompt, demo_method, demo_width,
                                              sample_seed=737, cap=CFG["max_new_tokens"])
    demo_rows.append({"method": decoder_label(demo_method, demo_width),
                      "translation": demo_pool[chosen]["text"],
                      "stopped_at_eos": demo_pool[chosen]["ended_eos"]})
print(NEW_SOURCE)
display(pd.DataFrame(demo_rows))
print("MBR samples for this source:")
display(pd.DataFrame({"sample": range(1, len(demo_pool) + 1),
                      "candidate": [item["text"] for item in demo_pool],
                      "selected": [i == chosen for i in range(len(demo_pool))]}))

### Collect the files for your team

The archive includes the settings and software versions, data and tokenizer fingerprints, shared splits, checkpoint and training history, quantitative prediction, test IDs, per-source candidates, per-run metrics and signatures, quality–cost plot, and error review sheet. The notebook file itself can be saved from Colab's **File → Download → Download .ipynb**.

For the project report, quote the parameter count and both the full and selected-checkpoint budgets from `training_summary.csv`. State which token definition you used. For Q3, name the data split, number of scored sentences, hardware, beam width, $k$ values, generation cap, and seeds. Quote both a quality difference and its latency cost. The spread over decoding seeds describes Monte Carlo variability conditional on this checkpoint and these sentences. Error categories require reading the examples.

In [ ]:
import shutil
from IPython.display import FileLink

archive = Path(shutil.make_archive(str(OUT.parent / f"{OUT.name}_results"), "zip", root_dir=OUT))
print(f"Results directory: {OUT}")
print(f"Downloadable ZIP: {archive} ({archive.stat().st_size / 2**20:.1f} MiB)")
if "google.colab" in sys.modules:
    from google.colab import files
    # Set True and rerun just this cell to download the ZIP in your browser.
    DOWNLOAD_ZIP = False
    if DOWNLOAD_ZIP:
        files.download(str(archive))
else:
    display(FileLink(str(archive)))